# WP31 — Curriculum Generator
## Layer 12: Adaptive Difficulty Scheduling via Zone of Proximal Development

---

This notebook demonstrates **WP31: Curriculum Generator**, the final layer of the
Prometheus synthesis stack. WP31 closes the **static-difficulty gap** in WP30: rather
than presenting puzzles from a fixed distribution, it tracks per-category accuracy
and selects puzzles in the learner's *Zone of Proximal Development* (ZPD).

### The Gap WP31 Closes

`WorldModelCRLS` (WP30) presents puzzles from a fixed distribution every generation:
1. **Wasted capacity**: easy puzzles solved 100% contribute zero learning signal
2. **Premature difficulty**: hard puzzles before prerequisites = noisy gradients

### WP31 Solution: ZPD-Based Curriculum

| Component | Role |
|-----------|------|
| `PuzzleCategory` | Per-category accuracy tracker |
| `CurriculumScheduler` | Selects category in ZPD (accuracy ∈ [0.40, 0.80]) |
| `CurriculumRecord` | Per-generation audit: category chosen, ZPD status |
| `CurriculumCRLS` | WorldModelCRLS + WP31 adaptive curriculum layer |

### Selection Logic
```
if any category in ZPD [0.40, 0.80]:  → pick highest difficulty in ZPD
elif all categories too easy (>0.80):  → push to hardest
elif all categories too hard (<0.40):  → pull to easiest
else:                                  → round-robin
```

### Theoretical Grounding
> *"Learning is most efficient when task difficulty is just beyond the learner's
> current level."* — L.S. Vygotsky (1978)

Runtime: **~8 min** (no GPU required)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings; warnings.filterwarnings('ignore')
import time, random, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from prometheus.wp31_curriculum_generator import (
    CurriculumCRLS, CurriculumScheduler, PuzzleCategory,
    CurriculumRecord,
    verify_wp31_exit_criteria,
)
from prometheus.wp22_bandit_exploration import BanditMode
from prometheus.wp17_crls_synthesis import SynthesisAction
from prometheus.environments.go import GoBoard

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP31 imports OK.')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
QUICK_MODE      = True
N_GENERATIONS   = 15 if QUICK_MODE else 35
PUZZLES_PER_GEN = 25 if QUICK_MODE else 60
BOARD_SIZE      = 9
ZPD_LO          = 0.40   # lower ZPD bound
ZPD_HI          = 0.80   # upper ZPD bound

# Three puzzle categories with increasing difficulty
CATEGORIES = [
    PuzzleCategory(name='EASY',   difficulty=0.20),
    PuzzleCategory(name='MEDIUM', difficulty=0.55),
    PuzzleCategory(name='HARD',   difficulty=0.90),
]

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Generations: {N_GENERATIONS} | ZPD: [{ZPD_LO}, {ZPD_HI}]')
print('Puzzle categories:')
for cat in CATEGORIES:
    print(f'  {cat.name:<8}  difficulty={cat.difficulty:.2f}')

---
## Section 1 — CurriculumScheduler: Standalone Demonstration

Before running the full stack, we demonstrate the `CurriculumScheduler` in isolation
to show how it selects categories based on the learner's current accuracy profile.

In [ ]:
# ── 2. Demonstrate CurriculumScheduler in isolation ──────────────────────────
from prometheus.wp31_curriculum_generator import CurriculumScheduler, PuzzleCategory

demo_cats = [
    PuzzleCategory(name='EASY',   difficulty=0.20),
    PuzzleCategory(name='MEDIUM', difficulty=0.55),
    PuzzleCategory(name='HARD',   difficulty=0.90),
]
scheduler = CurriculumScheduler(demo_cats, zpd_lo=ZPD_LO, zpd_hi=ZPD_HI)

# Simulate a learning progression
print('Simulated curriculum progression:')
print(f'  {"Phase":<30} {"Selected":<10} {"EASY":<8} {"MEDIUM":<10} {"HARD"}')
print('  ' + '-' * 70)

phases = [
    ('Start (no data yet)',                {})                                              ,
    ('All hard (<0.40)',                   {'EASY': 0.20, 'MEDIUM': 0.15, 'HARD': 0.05}   ),
    ('EASY mastered, MEDIUM in ZPD',       {'EASY': 0.90, 'MEDIUM': 0.60, 'HARD': 0.10}   ),
    ('MEDIUM in ZPD, HARD in ZPD',         {'EASY': 0.95, 'MEDIUM': 0.70, 'HARD': 0.50}   ),
    ('All mastered (>0.80)',               {'EASY': 0.95, 'MEDIUM': 0.85, 'HARD': 0.82}   ),
]

for phase_name, acc_map in phases:
    # Fresh scheduler for each scenario
    cats = [
        PuzzleCategory(name='EASY', difficulty=0.20),
        PuzzleCategory(name='MEDIUM', difficulty=0.55),
        PuzzleCategory(name='HARD', difficulty=0.90),
    ]
    sched = CurriculumScheduler(cats, zpd_lo=ZPD_LO, zpd_hi=ZPD_HI)
    for cat in cats:
        if cat.name in acc_map:
            cat.update(acc_map[cat.name])
    chosen = sched.select()
    e_acc = acc_map.get('EASY',   '---')
    m_acc = acc_map.get('MEDIUM', '---')
    h_acc = acc_map.get('HARD',   '---')
    print(f'  {phase_name:<30} → {chosen.name:<10} {str(e_acc):<8} {str(m_acc):<10} {h_acc}')

print()
print('ZPD selection logic verified.')

---
## Section 2 — CurriculumCRLS: Full 12-Layer Stack

In [ ]:
# ── 3. Puzzle factory (difficulty-stratified) ────────────────────────────────
def make_easy_puzzle(board_size, rng):
    """Simple 1-liberty atari — easy to capture."""
    board = GoBoard(size=board_size)
    cx = board_size // 2
    board.board[cx, cx] = GoBoard.BLACK
    libs = [(cx-1, cx), (cx+1, cx), (cx, cx-1), (cx, cx+1)]
    libs = [(r, c) for r, c in libs if board.is_on_board(r, c)]
    for r, c in libs[:-1]:
        board.board[r, c] = GoBoard.WHITE
    target = libs[-1]
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, target

def make_medium_puzzle(board_size, rng):
    """3-stone group atari."""
    board = GoBoard(size=board_size)
    cx = board_size // 2
    stones = [(cx, cx), (cx, cx+1), (cx+1, cx)]
    for r, c in stones:
        if board.is_on_board(r, c) and board.board[r, c] == GoBoard.EMPTY:
            board.board[r, c] = GoBoard.BLACK
    all_libs = set()
    for r, c in stones:
        for nr, nc in board.get_neighbors(r, c):
            if board.board[nr, nc] == GoBoard.EMPTY:
                all_libs.add((nr, nc))
    libs = list(all_libs); rng.shuffle(libs)
    for r, c in libs[:-1]: board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, libs[-1]

def make_hard_puzzle(board_size, rng):
    """Territory-based puzzle — harder to evaluate."""
    board = GoBoard(size=board_size)
    for r, c in [(0, 0), (0, board_size-1), (board_size-1, 0)]:
        board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.BLACK
    return board, GoBoard.BLACK, (board_size-1, board_size-1)

def evaluate_move(board, move, correct_move, player):
    if move == correct_move: return True
    if board.is_legal_move(move[0], move[1], player):
        return len(board.would_capture(move[0], move[1], player)) > 0
    return False

PUZZLE_FACTORIES = {
    'EASY':   make_easy_puzzle,
    'MEDIUM': make_medium_puzzle,
    'HARD':   make_hard_puzzle,
}

def generate_category_puzzles(category_name, n, board_size, seed):
    rng = np.random.default_rng(seed)
    return [PUZZLE_FACTORIES[category_name](board_size, rng) for _ in range(n)]

print('Difficulty-stratified puzzle factory OK.')

In [ ]:
# ── 4. Instantiate CurriculumCRLS ─────────────────────────────────────────────
STRATEGIES = ['EASY', 'MEDIUM', 'HARD', 'MIXED']

stack = CurriculumCRLS(
    strategies         = STRATEGIES,
    bandit_mode        = BanditMode.UCB1,
    curriculum_categories = CATEGORIES,
    zpd_lo             = ZPD_LO,
    zpd_hi             = ZPD_HI,
)

print('CurriculumCRLS (12 layers) instantiated.')
print('  Layers: WP17→WP19→WP20→WP21→WP22→WP23→WP24→WP25→WP26→WP27→WP30→WP31')
print(f'  Categories: {[c.name for c in CATEGORIES]}')
print(f'  ZPD: [{ZPD_LO}, {ZPD_HI}]')

accuracies, curriculum_records, selected_categories = [], [], []

In [ ]:
# ── 5. Main experiment loop ─────────────────────────────────────────────────
print('=' * 80)
print(f'  Gen  SelectedCat  Acc     ZPD_cats  MeanZPD_acc  DifficultyEmitted')
print('=' * 80)

for gen in range(N_GENERATIONS):
    # CurriculumCRLS will select the category internally; we provide puzzles for all
    # categories and let the stack choose which to use
    all_puzzles = {
        cat.name: generate_category_puzzles(cat.name, PUZZLES_PER_GEN, BOARD_SIZE,
                                             seed=gen * 137 + ord(cat.name[0]))
        for cat in CATEGORIES
    }

    # CurriculumCRLS expects a dict of category→puzzles; selects internally
    acc = stack.run_generation_curriculum(all_puzzles, evaluate_fn=evaluate_move)
    twelve_tuple = stack.end_of_generation()
    curr_rec = twelve_tuple[-1]   # CurriculumRecord is last element

    accuracies.append(acc)
    curriculum_records.append(curr_rec)
    selected_categories.append(curr_rec.selected_category if curr_rec else 'UNKNOWN')

    zpd_n    = curr_rec.n_zpd_categories if curr_rec else 0
    zpd_acc  = curr_rec.mean_zpd_accuracy if curr_rec else 0.0
    diff_out = curr_rec.difficulty_emitted if curr_rec else 0.0
    sel_cat  = curr_rec.selected_category  if curr_rec else '?'

    print(
        f'  {gen:3d}  {sel_cat:<12} {acc:.3f}   '
        f'{zpd_n}         '
        f'{zpd_acc:.3f}        '
        f'{diff_out:.2f}'
    )

print('=' * 80)
print(f'Mean accuracy: {np.mean(accuracies):.3f}')

In [ ]:
# ── 6. Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
gens = list(range(N_GENERATIONS))

cat_colors = {'EASY': '#4CAF50', 'MEDIUM': '#FF9800', 'HARD': '#f44336', 'UNKNOWN': '#9E9E9E'}

# A: Accuracy with category-coded background
ax = axes[0, 0]
for g, cat in enumerate(selected_categories):
    ax.axvspan(g - 0.5, g + 0.5, alpha=0.25, color=cat_colors.get(cat, '#eee'))
ax.plot(gens, accuracies, 'k-o', linewidth=2.5, markersize=7, zorder=3)
ax.axhline(np.mean(accuracies), color='grey', linestyle='--', alpha=0.7,
           label=f'Mean={np.mean(accuracies):.3f}')
# Legend for categories
handles = [mpatches.Patch(color=c, label=n, alpha=0.6) for n, c in cat_colors.items() if n != 'UNKNOWN']
ax.legend(handles=handles + ax.get_legend_handles_labels()[0][-1:], fontsize=9)
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy (background = selected category)', fontweight='bold')
ax.set_ylim(0, 1.05)

# B: Category selection over time
ax2 = axes[0, 1]
cat_numeric = {'EASY': 1, 'MEDIUM': 2, 'HARD': 3}
cat_values  = [cat_numeric.get(c, 0) for c in selected_categories]
scatter_colors = [cat_colors.get(c, '#9E9E9E') for c in selected_categories]
ax2.scatter(gens, cat_values, c=scatter_colors, s=100, zorder=3, edgecolors='black', linewidths=0.5)
ax2.step(gens, cat_values, where='mid', color='grey', alpha=0.4, linewidth=1.5)
ax2.set_yticks([1, 2, 3]); ax2.set_yticklabels(['EASY', 'MEDIUM', 'HARD'])
ax2.set_xlabel('Generation'); ax2.set_ylabel('Selected category')
ax2.set_title('Curriculum Selection over Time\n(ZPD guides: EASY→MEDIUM→HARD progression)', fontweight='bold')

# C: Per-category accuracy evolution
ax3 = axes[1, 0]
for cat in CATEGORIES:
    history = cat._acc_history
    if history:
        ax3.plot(range(len(history)), history, 'o-', linewidth=2,
                 color=cat_colors[cat.name], label=f'{cat.name} (final={cat.mean_accuracy:.2f})',
                 markersize=5)
ax3.axhspan(ZPD_LO, ZPD_HI, alpha=0.12, color='blue', label=f'ZPD [{ZPD_LO}, {ZPD_HI}]')
ax3.axhline(ZPD_LO, color='blue', linestyle=':', alpha=0.6)
ax3.axhline(ZPD_HI, color='blue', linestyle=':', alpha=0.6)
ax3.set_xlabel('Observations per category'); ax3.set_ylabel('Mean accuracy')
ax3.set_title('Per-Category Accuracy Evolution\n(blue band = ZPD sweet spot)', fontweight='bold')
ax3.legend(fontsize=9); ax3.set_ylim(0, 1.05)

# D: Summary
ax4 = axes[1, 1]
ax4.axis('off')
cat_counts = Counter(selected_categories)
summary_text = (
    'WP31 Curriculum Generator — Summary\n'
    '═════════════════════════════════════\n\n'
    f'  Total generations: {N_GENERATIONS}\n'
    f'  ZPD: [{ZPD_LO}, {ZPD_HI}]\n\n'
    'Category selection counts:\n'
    + ''.join(f'  {n:<8}: {c}x ({c/N_GENERATIONS:.0%})\n'
               for n, c in sorted(cat_counts.items())) +
    '\nPer-category final accuracy:\n'
    + ''.join(f'  {cat.name:<8}: {cat.mean_accuracy:.3f}\n'
               for cat in CATEGORIES) +
    f'\n  Mean overall: {np.mean(accuracies):.3f}\n\n'
    'Vygotsky (1978) ZPD:\n'
    '  Learning is most efficient\n'
    '  just beyond current level.\n\n'
    'Bengio et al. (2009):\n'
    '  Curriculum ordering improves\n'
    '  convergence + final perf.'
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=9.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(
    'WP31: Curriculum Generator — Prometheus v0\n'
    'Layer 12 (Final): Adaptive difficulty scheduling via Zone of Proximal Development',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('wp31_curriculum_generator.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to wp31_curriculum_generator.png')

In [ ]:
# ── 7. Verify WP31 exit criteria ─────────────────────────────────────────────
results = verify_wp31_exit_criteria(stack)
print('WP31 Exit Criteria Verification')
print('=' * 50)
all_pass = True
for criterion, passed in results.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed: all_pass = False
print()
if all_pass:
    print('All WP31 exit criteria satisfied.')
    print('The curriculum generator is adaptively selecting puzzle categories')
    print('based on the learner\'s Zone of Proximal Development.')
else:
    print('Some criteria not yet met — run more generations.')

---
## Conclusions

**Curriculum learning** (Vygotsky 1978, Elman 1993, Bengio et al. 2009) is the final
and highest-level meta-adaptation in the Prometheus stack. WP31 closes the loop on
WP30's world model by asking not just *how the system should act* but *what it should
practice*. The ZPD-based selector ensures the system always operates at the frontier
of its competence — the most efficient regime for learning.

### The Complete 12-Layer Stack

```
Layer 12  CurriculumScheduler (WP31) — ZPD-based puzzle selection
Layer 11  WorldModelCRLS (WP30)      — neural transition model
Layer 10  InvariantCRLS (WP27)       — formal safety invariants
Layer 9   TaskDecomposer (WP26)      — hierarchical decomposition
Layer 8   EWC guard (WP25)           — catastrophic forgetting
Layer 7   EnsembleDistiller (WP24)   — JSD uncertainty gate
Layer 6   PolicyDistiller (WP23)     — student distillation
Layer 5   BanditPolicy (WP22)        — explore/exploit
Layer 4   MetaGradientOptimiser (WP21)
Layer 3   RolloutPlanner (WP20)
Layer 2   CausalActionEvaluator (WP19)
Layer 1   WP17 heuristic
```

### References
- Vygotsky, L.S. (1978). *Mind in Society: The Development of Higher Psychological Processes*. Harvard UP.
- Elman, J.L. (1993). Learning and development in neural networks: The importance of starting small. *Cognition*, 48(1), 71–99.
- Bengio, Y. et al. (2009). Curriculum learning. *ICML*.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine. *Advances in Computers*, 6, 31–88.